In [36]:
from abc import ABC, abstractmethod
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        pass

In [37]:
import random
class NakliLLM(Runnable):
    def __init__(self):
        print('LLM created')

    def invoke(self, prompt):
        response_list = ['Delhi is the capital of India','IPL is a cricket league','AI means artificial intelligence']
        return {'response': random.choice(response_list)}


In [38]:
llm = NakliLLM()

LLM created


In [40]:
llm.invoke('what is the capital of india?')

{'response': 'Delhi is the capital of India'}

In [41]:
class NaklPrompttemplate(Runnable):
    def __init__(self, template, input_variables):
         self.template = template
         self.input_variables = input_variables
        
    def invoke(self, input_dict):
        return self.template.format(**input_dict)
    
    

In [42]:
template = NaklPrompttemplate(template='Write a {length} poem about {topic}', 
                              input_variables=['length','topic'])
prompt = template.invoke({'length':'short','topic':'india'})

In [57]:
llm = NakliLLM()
llm.invoke(prompt)


LLM created


{'response': 'AI means artificial intelligence'}

In [58]:
class NakliLLMchain:

    def __init__(self, prompt, llm):
        self.prompt = prompt
        self.llm = llm
    
    def run(self, input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)

        return result['response']

In [59]:
template = NaklPrompttemplate(template='Write a {length} poem about {topic}', 
                              input_variables=['length','topic'])

In [60]:
llm = NakliLLM()


LLM created


In [48]:
chain = NakliLLMchain(llm, template)


In [49]:
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list
    
    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [52]:
chain = RunnableConnector([template, llm])
chain.invoke({'length':'long','topic':'india'})


{'response': 'Delhi is the capital of India'}

In [53]:
template1 = NaklPrompttemplate(
    template='Write a joke about {topic}', 
    input_variables=['topic']
    )

template2 = NaklPrompttemplate(
    template='Explain the following joke \n {response}', 
    input_variables=['response']
    )

In [54]:
class NakilStringoutputparser(Runnable):
    def __init__(self):
        pass

    def invoke(self,input_data):
        return input_data['response']

In [55]:
parser = NakilStringoutputparser()

In [65]:
chain1 = RunnableConnector([template1,llm])
chain2 = RunnableConnector([template2,llm, parser])

In [66]:
final_chain = RunnableConnector([chain1,chain2])
final_chain.invoke({'topic':'Dance'})

'IPL is a cricket league'